# Related Papers:
- https://aclanthology.org/2024.acl-short.2.pdf
- https://aclanthology.org/2024.emnlp-main.171.pdf
- https://arxiv.org/pdf/2408.09174v1
- https://aclanthology.org/2025.findings-emnlp.638.pdf
- https://aclanthology.org/2025.emnlp-main.363.pdf

In [1]:
%load_ext autoreload
%autoreload 2

import os
import sys
from dotenv.main import load_dotenv

load_dotenv()
REPO_PATH = os.getenv('REPO_PATH')

sys.path.append(REPO_PATH)

In [2]:
import json

def load_json(file_path):
    with open(file_path, "r", encoding="utf-8") as file:
        return json.load(file)

def save_json(file_name, file):
    with open(os.path.join(REPO_PATH, "data_files", f"{file_name}.json"), "w", encoding="utf-8") as out_file:
        json.dump(file, out_file, indent=4)

def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]

# Data Creation:

## Havest Wikipedia pages with tables:

In [91]:
from knowledge_maze.create_dataset.create_questions.wikipedia_mining import WikiHarvester
# Initialize Wikipedia harvester
harvester = WikiHarvester()
# Run Phase 1 - Harvesting titles based on keywords
harvester.run_complete_harvest()
# Run Phase 2 - Filtering pages for tables
await harvester.filter_pages_async()

Harvesting Pages By Keywords:   0%|          | 0/233 [00:00<?, ?keyword/s]


TOTAL UNIQUE TITLES COLLECTED: 167913



Method,Keyword,Count
Prefix,List of cities,1537
Prefix,List of countries,1278
Prefix,List of towns,501
Prefix,List of villages,483
Prefix,List of districts,375
Prefix,List of municipalities,1021
Prefix,List of communes,194
Prefix,List of counties,259
Prefix,List of provinces,84
Prefix,List of territories,22


Filtering Pages with Tables:   0%|          | 0/167913 [00:00<?, ?page/s]


TOTAL PAGES WITH TABLES COLLECTED: 62306



In [93]:
save_json("table_titles", harvester.valid_titles)

In [3]:
table_titles = load_json("C:/Users/Daniel/Desktop/KnowledgeMaze/data_files/table_titles.json")

## Parse and filter Wikipedia tables:

In [271]:
from knowledge_maze.create_dataset.create_questions.table_processing import AsyncWikiExtractor, TablePostProcessor

extractor = AsyncWikiExtractor(output_file=os.path.join(REPO_PATH, "data_files/final_tables.jsonl"))
validated_tables_file_name = await extractor.run_with_progress(table_titles)
validated_tables = load_jsonl(os.path.join(REPO_PATH, "data_files/final_tables.jsonl"))

processor = TablePostProcessor(os.path.join(REPO_PATH, "data_files/final_tables.jsonl"))
cleaned_data = processor.process()

Pre-processing: cluster similar pages and discard similarities ---> 6426 pages discarded.


Extracting Tables:   0%|          | 0/55880 [00:00<?, ?Page/s]

Starting Extraction: 55880 titles | Async Limit: 20 | CPU Workers: 19
TOTAL PAGES WITH VALID TABLES: 32076 OUT OF 55880 (57.4%)


Filtering & Cleaning:   0%|          | 0/94101 [00:00<?, ?it/s]


════════════════════════════════════════════
📊 TABLE PROCESSING SUMMARY (Limit: 12)
════════════════════════════════════════════
Total tables scanned:      94101
Dropped (Bad Headers):     8435
Dropped (Cell too long):   2841
Dropped (Repeat Rows):     1783
Short tables found:        39837
Unique source pages:       16940
Final sampled (1/page):    16940
════════════════════════════════════════════

------------------------------------------------------------------------------------------------------------------------------------------------------
                                                                  ↓ EXAMPLE TABLES ↓                                                                  
------------------------------------------------------------------------------------------------------------------------------------------------------

══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════

,Rank,Player,Career,Caps,Goals
0,1,Christine Sinclair,2000–2023,331,190
1,2,Sophie Schmidt,2005–2023,226,20
2,3,Diana Matheson,2003–2021,206,19
3,4,Desiree Scott,2010–2025,188,0
4,5,Rhian Wilkinson,2003–2017,181,7
5,6,Kadeisha Buchanan,2013–present,154,6
6,7,Ashley Lawrence,2013–present,151,8
7,7,Jessie Fleming,2013–present,151,20
8,9,Brittany Baxter,2002–2014,132,5
9,9,Andrea Neil,1991–2007,132,24



══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
Manchester City F.C. 3–2 Queens Park Rangers F.C. > Match > Statistics
══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════


,Statistic,Manchester City,Queens Park Rangers
0,Goals scored,3,2
1,Possession,81.3%,18.7%
2,Shots on target,15,3
3,Shots off target,10,0
4,Blocked shots,19,0
5,Corner kicks,19,0
6,Fouls,4,7
7,Tackles,15,22
8,Offsides,1,1
9,Yellow cards,2,1



══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
Swinburne University of Technology > Notable people > Academics and staff
══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════


,Member,Executive team,Period
0,Pascale Quester,Vice-Chancellor,2020–present
1,Chris Pilgrim,Pro Vice-Chancellor (Academic),
2,Tara Magdalinski,Pro Vice-Chancellor (Education and Quality),
3,Sarah Maddison,Pro Vice-Chancellor (Academic Innovation and C...,
4,Bronwyn Fox,Deputy Vice-Chancellor (Research and Enterprise),
5,Beth Webster,Pro Vice-Chancellor (Research Policy and Impact),
6,Qing-Lon Han,Pro Vice-Chancellor (Research Quality),
7,Alan Kin-Tak Lau,Pro Vice-Chancellor (International Research De...,
8,Blair Kuys,Dean of the School of Design and Architecture,
9,Matthew Bailes,Director of the Data Science Research Institut...,


## Create Potential Entity Groups:

In [7]:
from knowledge_maze.create_dataset.create_questions.grouping_entities import TableGroupConstructor

group_constructor = TableGroupConstructor(random.sample(cleaned_data, 1000))
await group_constructor.get_groups()

🚀 Grouping 1000 tables using Entity-Focus logic...


Usage cost: 0.0000$ (Estimated: 0.00$):   0%|          | 0/1000 [00:00<?, ?it/s]


════════════════════════════════════════════════════════════════════════════════
                               📊 GROUPING REPORT                                
════════════════════════════════════════════════════════════════════════════════
  ✅ Valid Groups:   864
  ❌ Invalid Tables: 136
════════════════════════════════════════════════════════════════════════════════

✨ SAMPLE VALID NAMES:
  → Ports and harbours in Tanzania
  → Chart performance for Carver City in 2009
  → Filmfare Awards received by Atif Aslam

────────────────────────────────────────────────────────────────────────────────
                     🕵️ REJECTION DEBUGGER (Random Sample)                      
────────────────────────────────────────────────────────────────────────────────

[TABLE]: Mesa Grande Band of Diegueno Mission Indians > Reservation > Demographics (Mesa Grande Reservation, California – Racial and ethnic composition Note: the US Census treats Hispanic/Latino as an ethnic category. This table exclud

In [12]:
for sample in random.sample(group_constructor.valid_tables, 25):
    print(f"Group name: {sample['Group name']}")
    print(f"Wikipedia page: {sample['Wikipedia link']}")
    print(f"Entity column: {sample['Entity column']}")
    display(pd.DataFrame(sample['Table data']))
    print()
    print()

Group name: Rankings of Steve Williams in the 200 meters
Wikipedia page: https://en.wikipedia.org/wiki/Steve_Williams_(sprinter)#Rankings
Entity column: Year


,Year,World rank,US rank
0,1972,-,6th
1,1973,1st,1st
2,1974,6th,2nd
3,1975,2nd,1st
4,1976,3rd,2nd
5,1977,5th,2nd
6,1978,6th,2nd
7,1979,-,-
8,1980,4th,1st




Group name: Highest-grossing Hong Kong films of 2019
Wikipedia page: https://en.wikipedia.org/wiki/List_of_Hong_Kong_films_of_2019#Box_office
Entity column: Title


,Rank,Title,Distributor,Domestic gross
0,1,Integrity,Emperor Cinema,"HK$30,986,111"
1,2,Ip Man 4: The Finale,Mandarin Motion Pictures,"HK$28,983,160"
2,3,The White Storm 2: Drug Lords,Universe Films Distribution,"HK$24,373,233"
3,4,Still Human,Golden Scene Company Limited,"HK$19,201,830"
4,5,P Storm,Mandarin Motion Pictures,"HK$17,514,387"




Group name: Teams in the 1992–93 Manchester City F.C. season final league table
Wikipedia page: https://en.wikipedia.org/wiki/1992–93_Manchester_City_F.C._season#Final_league_table
Entity column: Team v t e


...

,Organization,Host city,Host country,Mission
0,Food and Agriculture Organization,Rome,Italy,Permanent Mission
1,International Civil Aviation Organization,Montreal,Canada,Permanent Mission
2,Organization of American States,"Washington, D.C.",United States,Permanent Mission
3,United Nations,New York City,United States,Permanent Mission
4,United Nations,Geneva,Switzerland,Permanent Mission
5,United Nations,Vienna,Austria,Permanent Mission
6,UNESCO,Paris,France,Permanent Delegation
7,World Trade Organization,Geneva,Switzerland,Permanent Mission




Group name: Judging components for figure skating disciplines at the 2022 Winter Olympics
Wikipedia page: https://en.wikipedia.org/wiki/Figure_skating_at_the_2022_Winter_Olympics_–_Team_event#Judging
Entity column: Discipline


,Discipline,Short program or Rhythm dance,Free skate or Free dance
0,Men,1.0,2.0
1,Women,0.8,1.6
2,Pairs,0.8,1.6
3,Ice dance,0.8,1.2




Group name: Highest-grossing Spanish films of 2008
Wikipedia page: https://en.wikipedia.org/wiki/List_of_Spanish_films_of_2008#Box_office
Entity column: Title


,Rank,Title,Distributor,Admissions,Domestic gross ( € )
0,1,The Oxford Murders ( Los crímenes de Oxford ),Warner Bros.,1421063,8201220.76
1,2,Mortadelo and Filemon. Mission: Save the Plane...,Zeta Audiovisual,1363275,7706810.77
2,3,Vicky Cristina Barcelona,Warner Bros.,1240343,7459286.88
3,4,Che ( Che. El argentino ),Hispano Foxfilm,1161635,6830886.78
4,5,Asterix at the Olympic Games ( Astérix en los ...,Tripictures,1051286,5905546.01
5,6,Chef's Special ( Fuera de carta ),Warner Bros.,898186,5107626.42
6,7,The Blind Sunflowers ( Los girasoles ciegos ),Alta Classics,710318,4073176.61
7,8,Transsiberian,Sociedad General de Cine,393453,2348860.83
8,9,The El Escorial Conspiracy ( La conjura de El ...,Sony,318580,1845819.84
9,10,Carlitos and the Field of Dreams ( Carlitos y...,Hispano Foxfilm,258345,1401541.26


## Create FanOut-style Questions out of the Entity Groups:

1. **Information Retrieval & Statistical Totals:**
    - **COUNT_TOTAL:** Identifying the total number of distinct entities that belong to the fully specified group described in the evidence ("The text describes five different hospitals founded in the 19th century. How many total hospitals are listed?")
    - **UNIQUE_DIVERSITY:** Counting the number of unique categorical attributes (e.g., different types, countries, or genres) present across the entities ("Across the six listed books, how many different publishers are represented?")
2. **Conditional Filtering & Grouped Logic:**
    - **FILTER_COUNT_CATEGORICAL:** Identifying a specific non-numerical attribute and counting how many entities match a certain value ("Out of the eight projects mentioned, how many are classified as 'In Progress'?")
    - **FILTER_COUNT_RANGE:** Counting entities that fall within a specific numerical or temporal boundary ("How many of the listed athletes were born between 1990 and 1995?")
    - **GROUPED_LOOKUP:** Filtering entities into a specific sub-group first, then performing a secondary identification or count based on another attribute ("Among the cities located in 'North America,' how many have a population exceeding 5 million?")
3. **Numerical Ranking & Extrema Analysis:**
    - **EXTREME_NUMERICAL:** Identifying the specific entity that holds the maximum/minimum value ("Which of the four listed technology companies had the highest revenue in the final quarter?")
    - **EXTREME_TEMPORAL:** Identifying the specific entity that holds the earliest/latest date ("Identify the three landmarks that were completed or inaugurated the earliest.")
    - **TOP_K_RANKING:** Sorting entities based on a metric and identifying a specific subset (e.g., the top 3) in order ("List the three cars with the fastest 0-60 mph times from the text in descending order.")
4. **Comparative & Distribution Analysis:**
    - **FREQUENCY_MODE:** Analyzing a categorical attribute across all evidence points to find which value appears most frequently ("Based on the descriptions of the ten software engineers, which primary programming language is cited most often?")
    - **COMPARATIVE_DIFFERENCE:** Selecting two specific entities and calculating the numerical difference between their shared attributes ("What is the difference in seating capacity between the Grand Theatre and the City Playhouse?")
    - **AGGREGATION:** Calculating the sum or average of a numerical attribute across specified entities to derive aggregate metrics ("Based on the evidence for the four product launches, what is the average initial production cost for the three models released in 2024?")

### Rule-Based Strategy Candidate Selection:

In [241]:
from knowledge_maze.create_dataset.create_questions.question_generation import StrategySelector

selector = StrategySelector(group_constructor.valid_tables)
selector.get_strategies()

Sanitizing tables:   0%|          | 0/864 [00:00<?, ?it/s]

Slicing strategy views:   0%|          | 0/864 [00:00<?, ?it/s]


✅ Strategy Selection Complete.

═══════════════════════════════════════════════════════
📊 STRATEGY SELECTION SUMMARY
-------------------------------------------------------
✅ Groups Passed:     842
❌ Groups Discarded:  22
📈 Success Rate:      97.5%
-------------------------------------------------------
Strategy Distribution (Top 3 per table):
  - UNIQUE_DIVERSITY          : 497
  - EXTREME_TEMPORAL          : 365
  - FREQUENCY_MODE            : 258
  - CATEGORICAL_FILTER_LOOKUP : 225
  - COUNT_TOTAL               : 222
  - EXTREME_NUMERICAL         : 140
  - FILTER_COUNT_TEMPORAL     : 129
  - AGGREGATION               : 123
  - TOP_K_RANKING             : 118
  - COMPARATIVE_DIFFERENCE    : 109
  - FILTER_COUNT_NUMERICAL    : 106
═══════════════════════════════════════════════════════



In [225]:
# EXTREME_NUMERICAL         : Success/Faliure
# EXTREME_TEMPORAL          : 
# COUNT_TOTAL               : 
# UNIQUE_DIVERSITY          : 
# CATEGORICAL_FILTER_LOOKUP : 
# FILTER_COUNT_TEMPORAL     : 
# FREQUENCY_MODE            : 
# EXTREME_NUMERICAL         : 
# AGGREGATION               : 
# FILTER_COUNT_NUMERICAL    : 
# TOP_K_RANKING             : 
# COMPARATIVE_DIFFERENCE    : 

In [242]:
selector.debug_strategy_samples("AGGREGATION")

Found 123 total tables with this strategy. Showing 5 random samples.



🔗 Source: https://en.wikipedia.org/wiki/Strong_Openweight_Tag_Team_Championship#Combined_reigns


,Team,No. of reigns,Combined defenses
0,Guerrillas of Destiny ( Hikuleo and El Phantas...,2,5
1,United Empire ( TJP and Templario ),1,2
2,TMDK ( Mikey Nicholls and Shane Haste ),2,3
3,The Motor City Machine Guns ( Alex Shelley and...,1,3
4,World Class Wrecking Crew ( Royce Isaacs and J...,1,2
5,Aussie Open ( Kyle Fletcher and Mark Davis ),2,4
6,Bullet Club War Dogs ( Alex Coughlin and Gabe ...,1,0
7,Los Hermanos Chávez † ( Ángel de Oro and Niebl...,1,1
8,Grizzled Young Veterans ( James Drake and Zack...,1,1
9,Bishamon ( Hirooki Goto and Yoshi-Hashi ),1,0


🔗 Source: https://en.wikipedia.org/wiki/List_of_stadiums_in_Wales_by_capacity#Capacity_of_at_least_10,000


,Stadium,Seated capacity
0,Principality Stadium,73931
1,Cardiff City Stadium,33280
2,Swansea.com Stadium,21000
3,Sophia Gardens,16000
4,Parc y Scarlets,14000
5,Cardiff Arms Park,12500
6,STōK CaeRas (Racecourse Ground),10771


...

,Name,Capacity ( MW )
0,Keechelus Dam,0
1,Easton Diversion Dam,0
2,Unnamed Diversion Dam,0
3,Roza Diversion Dam,12
4,Wapato Dam,0
5,Sunnyside Dam,0
6,Prosser Dam,0
7,Wannawish Dam,0


🔗 Source: https://en.wikipedia.org/wiki/1931_International_Cross_Country_Championships#Team_Results


,Team,Points
0,Jack Winfield Tom Evenson Jack Potts Jack Hold...,32
1,Robbie Sutherland John Suttie Smith Jimmy Wood...,102
2,Henri Lahitte Robert Loiseau Marcel Michot Alb...,102
3,Tim Smythe Thomas Kinsella F. Mills J.C. McInt...,112
4,Harry Gallivan Danny Phillips Ernie Thomas Tom...,184
5,Oscar van Rumst Theo Meersman Julien Serwy Jos...,201


🔗 Source: https://en.wikipedia.org/wiki/2018_USA_Team_Handball_Nationals_–_men's_open_division#Results


,Team,W,L,GF,GA,Pts
0,Los Angeles THC (19),4,1,116,86,8
1,Carolina Blue (12),3,2,119,102,6
2,Long Island Tigers (NV),3,2,108,121,6
3,West Point Gold (24),2,3,118,128,4
4,San Francisco CalHeat 2 (NV),2,3,82,89,4
5,Minnesota (NR),1,4,93,110,2


### Model-Based Strategy Validation & Question Generation:

In [214]:
from knowledge_maze.create_dataset.create_questions.question_generation import TableSanitizer, StrategySelector, QuestionGenerator

sanitizer = TableSanitizer(group_constructor.valid_tables)
sanitizer.process_tables()

selector = StrategySelector(sanitizer.sanitized_tables)
_ = selector.get_strategies()

# generator = QuestionGenerator(random.sample(selector.updated_data, 200), 
#                               strategy_definitions=load_json(os.path.join(REPO_PATH, "knowledge_maze/create_dataset/create_questions/question_strategies.json")))
# _ = await generator.generate_questions()

Sanitizing tables:   0%|          | 0/864 [00:00<?, ?it/s]

Slicing strategy views:   0%|          | 0/864 [00:00<?, ?it/s]


✅ Strategy Selection Complete.

═══════════════════════════════════════════════════════
📊 STRATEGY SELECTION SUMMARY
-------------------------------------------------------
✅ Groups Passed:     842
❌ Groups Discarded:  22
📈 Success Rate:      97.5%
═══════════════════════════════════════════════════════



In [218]:
selector.debug_strategy_samples("CATEGORICAL_FILTER_LOOKUP")

Found 225 total tables with this strategy. Showing 5 random samples.



🔗 Source: https://en.wikipedia.org/wiki/List_of_diplomatic_missions_in_the_San_Francisco_Bay_Area#Other_missions


,Country,City
0,Canada,Palo Alto
1,European Union,San Francisco
2,Hong Kong,San Francisco
3,Taiwan,San Francisco


🔗 Source: https://en.wikipedia.org/wiki/List_of_diplomatic_missions_of_the_Dominican_Republic#Current_missions


,Organization,Host country
0,Food and Agriculture Organization,Italy
1,International Civil Aviation Organization,Canada
2,Organization of American States,United States
3,United Nations,United States
4,United Nations,Switzerland
5,United Nations,Austria
6,UNESCO,France
7,World Trade Organization,Switzerland


...

,School,Suburb
0,Calrossy Anglican School – Brisbane Street Cam...,East Tamworth
1,Calrossy Anglican School – William Cowper Camp...,North Tamworth
2,Calrossy Anglican School – William Cowper Camp...,North Tamworth
3,Carinya Christian School,Calala
4,McCarthy Catholic College,North Tamworth
5,St Edward's Catholic School,South Tamworth
6,St Joseph's Catholic School,West Tamworth
7,St Mary's Campus of St Nicholas School,Oxley Vale
8,St Nicholas Catholic School,East Tamworth


🔗 Source: https://en.wikipedia.org/wiki/List_of_lighthouses_in_Malaysia#Sabah


,Lighthouse,Constituency
0,Pulau Kalampunian Lighthouse,Kudat
1,Pulau Mantanani Lighthouse,Kudat
2,Pulau Tiga Lighthouse,Papar
3,Pulau Kuraman Lighthouse,Labuan
4,Pulau Papan Lighthouse,Labuan
5,Batu Tinagat Lighthouse,Tawau
6,Pulau Berhala Lighthouse,Sandakan
7,Tanjung Trang Lighthouse,Lahad Datu


🔗 Source: https://en.wikipedia.org/wiki/Provinces_of_Korea#Provinces_of_Unified_Silla


,Province,Former kingdom
0,Yangju,Silla
1,Gangju,Silla
2,Sangju,Silla
3,Muju,Baekje
4,Jeonju,Baekje
5,Ungju,Baekje
6,Hanju,Goguryeo
7,Sakju,Goguryeo
8,Myeongju,Goguryeo


In [169]:
aaa = 0
for dt in selector.updated_data:
    if "FILTER_COUNT_TEMPORAL" in dt['Top strategies']:
        aaa += 1
print(aaa)

100


In [174]:
# EXTREME_NUMERICAL         : Success
# EXTREME_TEMPORAL          : Success
# COUNT_TOTAL               : Success
# UNIQUE_DIVERSITY          : Success
# CATEGORICAL_FILTER_LOOKUP : Faliure
# FILTER_COUNT_TEMPORAL     : Faliure
# FREQUENCY_MODE            :
# EXTREME_NUMERICAL         :
# AGGREGATION               :
# FILTER_COUNT_NUMERICAL    :
# TOP_K_RANKING             :
# COMPARATIVE_DIFFERENCE    :

import random
import pandas as pd

TARGET_STRATEGY = "FREQUENCY_MODE"
N = 5

matches = [
    sample
    for sample in generator.valid_results
    if sample.get("Selected strategy") == TARGET_STRATEGY
]

for sample in random.sample(matches, min(N, len(matches))):
    print(f"Group name: {sample['Group name']}")
    print(f"Strategy: {sample['Selected strategy']}")
    print(f"Question: {sample['Question']}")
    print(f"Wikipedia page: {sample['Wiki link']}")
    display(pd.DataFrame(sample['Relevant table data']))
    display(pd.DataFrame(sample['Full table data']))
    print("\n")

Group name: Members of the thirteenth Knesset and their replacements
Strategy: FREQUENCY_MODE
Question: What was the most common political party among the members of the thirteenth Knesset?
Wikipedia page: https://en.wikipedia.org/wiki/List_of_members_of_the_thirteenth_Knesset#List_of_members


,MK,Party
0,Avraham Ravitz,United Torah Judaism
1,Ariel Weinstein,Likud
2,Avraham Verdiger,United Torah Judaism
3,Moshe Gafni,United Torah Judaism
4,Saleh Saleem,Hadash
5,Haneh Hadad,Labor Party
6,Avraham Katz-Oz,Labor Party
7,Yosef Ahimeir,Likud
8,Nava Arad,Labor Party
9,Michael Ratzon,Likud


,MK,Replaced,Date,Party,Notes
0,Avraham Ravitz,Yitzhak Peretz,16 July 1992,United Torah Judaism,Represented Degel HaTorah when party split
1,Ariel Weinstein,Roni Milo,30 December 1993,Likud,
2,Avraham Verdiger,Menachem Porush,28 June 1994,United Torah Judaism,Represented Agudat Yisrael when party split
3,Moshe Gafni,Shmuel Halpert,28 June 1994,United Torah Judaism,Represented Degel HaTorah when party split
4,Saleh Saleem,Tawfiq Ziad,7 July 1994,Hadash,
5,Haneh Hadad,Avraham Burg,5 July 1995,Labor Party,
6,Avraham Katz-Oz,Mordechai Gur,16 July 1995,Labor Party,
7,Yosef Ahimeir,Haim Kaufman,7 August 1995,Likud,
8,Nava Arad,Yitzhak Rabin,5 November 1995,Labor Party,Left party to sit as an independent
9,Michael Ratzon,Ariel Weinstein,10 March 1996,Likud,




Group name: Aerobic microbiota species of the lower reproductive tract of women
Strategy: FREQUENCY_MODE
Question: What is the most common form of aerobic microbiota species found in the lower reproductive tract of women?
Wikipedia page: https://en.wikipedia.org/wiki/List_of_microbiota_species_of_the_lower_reproductive_tract_of_women#Normal_microbiota


,Genus,form
0,Aerococcus,cocci
1,Atopobium,bacillus
2,Dialister,bacillus
3,Eggerthella,bacillus
4,Finegoldia,
5,Gardnerella,coccobacilli
6,Megasphaera,
7,Mobiluncus,bacillus
8,Peptoniphilus,
9,Prevotella,


...

,Name,Type
0,Изопанхром (Isopanchrom),Print
1,Фото 32 (Foto 32),Print
2,Фото 65 (Foto 65),Print
3,Фото 130 (Foto 130),Print
4,Фото 250 (Foto 250),Print
5,ФН 64 (FN 64),Print
6,МЗ-3Л (MZ-3L),Positive
7,Фото 32 (Foto 32),Print
8,Фото 64 (Foto 64),Print
9,Фото 125 (Foto 125),Print


,Make,Name,Dates,Base,ISO,Process,Type,Details,Origin,Formats,Replaced by
0,Factory No. 8,Изопанхром (Isopanchrom),?–c1965,T,"80, 100",B&W,Print,General purpose panchromatic film. Made in thr...,USSR,935,?
1,Kazan / Tasma,Фото 32 (Foto 32),c1965–1987,T,40,B&W,Print,General purpose fine grain panchromatic film.,USSR,935,Foto 32
2,Kazan / Tasma,Фото 65 (Foto 65),c1965–1987,T,80,B&W,Print,General purpose panchromatic film.,USSR,"935, sheet film",Foto 64
3,Kazan / Tasma,Фото 130 (Foto 130),c1965–1987,T,160,B&W,Print,General purpose panchromatic film.,USSR,935,Foto 125
4,Tasma,Фото 250 (Foto 250),?–1987,T,320,B&W,Print,General purpose super panchromatic film. For e...,USSR,935,Foto 250
5,Tasma,ФН 64 (FN 64),?-c1993,T,64,B&W,Print,"""Non-standard"" film conforming to ASA scale. P...",USSR,935,Nothing
6,Tasma,МЗ-3Л (MZ-3L),?–1990,T,6,B&W,Positive,Slow speed orthochromatic positive film.,USSR,935,Nothing
7,Tasma,Фото 32 (Foto 32),1987–?,T,32,B&W,Print,General purpose fine grain panchromatic film.,USSR,935,Nothing
8,Tasma,Фото 64 (Foto 64),1987–?,T,64,B&W,Print,General purpose panchromatic film.,USSR,935,Nothing
9,Tasma,Фото 125 (Foto 125),1987–?,T,125,B&W,Print,General purpose panchromatic film.,USSR,935,Nothing




Group name: Israeli films of 1985 with unknown premiere dates
Strategy: FREQUENCY_MODE
Question: What is the most common genre of Israeli films released in 1985 with unknown premiere dates?
Wikipedia page: https://en.wikipedia.org/wiki/List_of_Israeli_films_of_1985#1985_releases


,Title,Genre
0,Lemon Popsicle VI ( Hebrew : אסקימו לימון 6 - ...,Comedy
1,51 Bar ( Hebrew : בר 51 ),"Crime , Drama"
2,"Za'am V'Tehilah ( Hebrew : זעם ותהילה , lit. ""...",Drama
3,"Banot ( Hebrew : בנות , lit. ""Girls"")",Comedy
4,"Ad Sof Halaylah ( Hebrew : עד סוף הלילה , lit....",Drama
5,"Ha-Me'ahev ( Hebrew : המאהב , lit. ""The Lover"")",Drama
6,"Shovrim ( Hebrew : שוברים , lit. ""Breaking"")",Comedy
7,"Koko Ben 19 ( Hebrew : קוקו בן 19 , lit. ""Koko...",Drama
8,"Irit, Irit ( Hebrew : עירית אירית )",Drama
9,"Kompot Na'alyim ( Hebrew : קומפוט נעליים , lit...",Comedy


,Premiere,Title,Director,Cast,Genre
0,?,Lemon Popsicle VI ( Hebrew : אסקימו לימון 6 - ...,Dan Wolman,"Yftach Katzur , Zachi Noy , Jonathan Sagall",Comedy
1,?,51 Bar ( Hebrew : בר 51 ),Amos Guttman,"Alon Aboutboul , Mosko Alkalai , Juliano Mer","Crime , Drama"
2,?,"Za'am V'Tehilah ( Hebrew : זעם ותהילה , lit. ""...",Avi Nesher,"Hana Azoulay-Hasfari , Juliano Mer",Drama
3,?,"Banot ( Hebrew : בנות , lit. ""Girls"")",Nadav Levitan,Hana Azoulay-Hasfari,Comedy
4,?,"Ad Sof Halaylah ( Hebrew : עד סוף הלילה , lit....",Eitan Green,Assi Dayan,Drama
5,?,"Ha-Me'ahev ( Hebrew : המאהב , lit. ""The Lover"")",Michal Bat-Adam,Michal Bat-Adam,Drama
6,?,"Shovrim ( Hebrew : שוברים , lit. ""Breaking"")",Avi Nesher,"Izhar Ashdot , Gali Atari , Sey Heiman , Alona...",Comedy
7,?,"Koko Ben 19 ( Hebrew : קוקו בן 19 , lit. ""Koko...",Dan Verete,Udi Cohen,Drama
8,?,"Irit, Irit ( Hebrew : עירית אירית )",Naftali Alter,"Irit Alter , Hanan Goldblatt , Ronnie Pinkovitz",Drama
9,?,"Kompot Na'alyim ( Hebrew : קומפוט נעליים , lit...","Yehuda Barkan , Ezra Shem-Tov",Yehuda Barkan,Comedy


In [194]:
from knowledge_maze.create_dataset.create_questions.question_validator import QuestionValidator

validator = QuestionValidator(load_json(os.path.join(REPO_PATH, "knowledge_maze/create_dataset/create_questions/question_strategies.json")))

ver = await validator.validate_samples(random.sample(generator.valid_results, 50))

Usage cost: 0.0000$ (Estimated: 0.00$):   0%|          | 0/50 [00:00<?, ?it/s]


📊 VALIDATION SUMMARY
Total Samples:   50
✅ Approved:     46
❌ Rejected:     4
📈 Pass Rate:    92.0%
--------------------------------------------------
📉 REJECTIONS BY STRATEGY:
  - EXTREME_TEMPORAL         : 3
  - TOP_K_RANKING            : 1
--------------------------------------------------

🔍 TOP REJECTION EXAMPLES:

1. Strategy: EXTREME_TEMPORAL
   Question: Who was the earliest winner of the Golden Globe Award for Best Screenplay in the 1950s?
   Data: Winners and nominees for the Golden Globe Award for Best Screenplay in the 1950s


,Film,Year
0,All About Eve †,1950
1,The Asphalt Jungle,1950
2,Sunset Boulevard ‡,1950
3,Bright Victory,1951
4,5 Fingers,1952
5,High Noon,1952
6,The Thief,1952
7,Lili,1953
8,Sabrina,1954


   🚩 Reason: The question asks for the 'earliest winner', but the provided table data does not specify which films were winners and which were nominees. It only lists 'Film' and 'Year'. Therefore, it's impossible to identify a 'winner' from the given data.

2. Strategy: EXTREME_TEMPORAL
   Question: Who was the first ambassador of Guinea-Bissau to China based on the earliest date of accreditation?
   Data: Ambassadors of Guinea-Bissau to China


,Ambassador,Designated/accredited
0,,"March 15, 1974"
1,,"May 31, 1990"
2,,"April 23, 1998"
3,Nicolau Dos Santos,"February 16, 2001"
4,Arafan Ansu Câmara,"September 3, 2010"
5,Malam Sambú,"October 12, 2012"


   🚩 Reason: The question asks for the 'first ambassador' based on the earliest date, but the earliest date in the provided data ('March 15, 1974') has an empty 'Ambassador' field. Therefore, the question cannot be answered with the given data.

3. Strategy: TOP_K_RANKING
   Question: What were the three teams in the relegation zone of the 1985–86 Football League First Division based on their points?
   Data: Teams in the 1985–86 Football League First Division relegation zone


,Team,Pts
0,Oxford United,42
1,Leicester City,42
2,Ipswich Town,41
3,Birmingham City,29
4,West Bromwich Albion,24


   🚩 Reason: The question asks for the 'three teams' but the provided 'Table Data' contains five teams, making it impossible to determine the top three without additional ranking information or a clear metric to sort by. The strategy 'TOP_K_RANKING' requires a clear numerical metric to rank, and while 'Pts' is present, the question doesn't specify how to use it to identify the 'three teams' from the given five, especially since two teams have the same 'Pts' value (42).

4. Strategy: EXTREME_TEMPORAL
   Question: What was the title of the earliest television performance of Marilyn Monroe?
   Data: Television performances of Marilyn Monroe


,Title,"Year, Title"
0,The Jack Benny Program,None
1,The Bob Hope Show,None
2,Person to Person,None
3,President Kennedy's Birthday Salute,None


   🚩 Reason: The question is unanswerable because the 'Year, Title' column, which is the temporal column, contains only null values. Therefore, it's impossible to determine the earliest performance.



In [155]:
# EXTREME_NUMERICAL         : 
# EXTREME_TEMPORAL          : 
# COUNT_TOTAL               : 
# UNIQUE_DIVERSITY          : 
# CATEGORICAL_FILTER_LOOKUP : 
# FILTER_COUNT_TEMPORAL     : 
# FREQUENCY_MODE            : Success
# EXTREME_NUMERICAL         :
# AGGREGATION               : Partial - made a prompt fix for the scope of Top X in the aggragation. Still - No avg for some reason.
# FILTER_COUNT_NUMERICAL    :
# TOP_K_RANKING             : Success (need to set k to random somehow).
# COMPARATIVE_DIFFERENCE    :

import random
import pandas as pd

TARGET_STRATEGY = "AGGREGATION"
N = 5

matches = [
    sample
    for sample in validator.approved_data
    if sample.get("Selected strategy") == TARGET_STRATEGY
]

for sample in random.sample(matches, min(N, len(matches))):
    print(f"Group name: {sample['Group name']}")
    print(f"Strategy: {sample['Selected strategy']}")
    print(f"Question: {sample['Question']}")
    print(f"Wikipedia page: {sample['Wiki link']}")
    display(pd.DataFrame(sample['Relevant table data']))
    # display(pd.DataFrame(sample['Full table data']))
    print("\n")

Group name: Champions in Players Series Snooker
Strategy: AGGREGATION
Question: What is the total number of championships won by players in the Champions in Players Series Snooker?
Wikipedia page: https://en.wikipedia.org/wiki/Players_Series#Statistics


,Player,Total
0,Neil Robertson,5
1,Ronnie O'Sullivan,4
2,Judd Trump,3
3,Shaun Murphy,2
4,Mark Allen,2
5,John Higgins,2
6,Stephen Maguire,1
7,Mark Williams,1
8,Kyren Wilson,1




Group name: Former Lynx Air Fleet
Strategy: AGGREGATION
Question: What is the total number of aircraft in the former Lynx Air fleet?
Wikipedia page: https://en.wikipedia.org/wiki/Lynx_Air#Fleet


,Aircraft,Total
0,Boeing 737-700,7
1,Boeing 737-800,1
2,Boeing 737-800,1
3,De Havilland Canada DHC-6 Twin Otter,2
4,Piper PA-30 Twin Comanche,1
5,Airbus A320-200,1




Group name: Men's standings in the 1997–98 FIS Cross-Country World Cup
Strategy: AGGREGATION
Question: What is the total number of points accumulated by all competitors in the **1997–98 FIS Cross-Country World Cup**?
Wikipedia page: https://en.wikipedia.org/wiki/1997–98_FIS_Cross-Country_World_Cup#Men's_standings


,after all 12 events,Points
0,Thomas Alsgaard,790
1,Bjørn Dæhlie,678
2,Vladimir Smirnov,431
3,Mikhail Botvinov,330
4,Jari Isometsä,316
5,Fulvio Valbusa,310
6,Mika Myllylä,308
7,Erling Jevne,292
8,Silvio Fauner,273
9,Mathias Fredriksson,268




Group name: Growth of British railways
Strategy: AGGREGATION
Question: What is the total number of miles of British railways constructed by 1860?
Wikipedia page: https://en.wikipedia.org/wiki/History_of_rail_transport#History_by_country


,Year,Total miles
0,1830,98
1,1835,338
2,1840,1498
3,1845,2441
4,1850,6621
5,1855,8280
6,1860,10433




Group name: Highest-grossing Hong Kong films of 2019
Strategy: AGGREGATION
Question: What is the total domestic gross of the highest-grossing films in Hong Kong for 2019?
Wikipedia page: https://en.wikipedia.org/wiki/List_of_Hong_Kong_films_of_2019#Box_office


,Title,Domestic gross
0,Integrity,"HK$30,986,111"
1,Ip Man 4: The Finale,"HK$28,983,160"
2,The White Storm 2: Drug Lords,"HK$24,373,233"
3,Still Human,"HK$19,201,830"
4,P Storm,"HK$17,514,387"


In [182]:
import random
import pandas as pd

def debug_selector_output(updated_data, target_strategy, num_samples=10):
    print(f"🔎 Scanning selector.updated_data for: {target_strategy}")
    
    # 1. Filter tables where the target strategy is present
    matches = [
        t for t in updated_data 
        if target_strategy in t.get('Top strategies', [])
    ]
    
    if not matches:
        print(f"❌ No tables found containing strategy: {target_strategy}")
        return

    print(f"🎯 Total Matches: {len(matches)}")
    print(f"🎲 Showing {min(num_samples, len(matches))} random samples:\n")
    
    # 2. Pick and display samples
    samples = random.sample(matches, min(num_samples, len(matches)))
    
    for i, table in enumerate(samples, 1):
        group_name = table.get('Group name', 'N/A')
        strats = table.get('Top strategies', [])
        
        print(f"--- SAMPLE {i} ---")
        print(f"TITLE: {group_name}")
        print(f"STRATEGIES: {strats}")
        print(f"DATA SNIPPET (First 2 rows):")
        data = table.get('Table data - cleaned', [])
        display(pd.DataFrame(data))
        print("-" * 40 + "\n")

# --- DEBUG RUN ---
# Change this string to 'FREQUENCY_MODE', 'UNIQUE_DIVERSITY', etc.
debug_selector_output(selector.updated_data, "FREQUENCY_MODE")

🔎 Scanning selector.updated_data for: FREQUENCY_MODE
🎯 Total Matches: 366
🎲 Showing 10 random samples:

--- SAMPLE 1 ---
TITLE: Animated series produced by Marvel Studios
STRATEGIES: ['FREQUENCY_MODE', 'UNIQUE_DIVERSITY', 'CATEGORICAL_FILTER_LOOKUP']
DATA SNIPPET (First 2 rows):


,Series,Released,Production partner(s),Distributor,Original network
0,Silver Surfer,1998,Saban Entertainment AKOM Marvel Entertainment ...,Saban International,Fox Kids
1,Spider-Man Unlimited,1999–2001,Saban Entertainment Koko Enterprises Dong Yang...,Saban International,Fox Kids
2,The Avengers: United They Stand,1999–2000,Saban Entertainment Fox Family Worldwide Saero...,Saban International,Fox Kids
3,X-Men: Evolution,2000–2003,Film Roman,Warner Bros. Television Distribution Marvel En...,Kids' WB
4,Fantastic Four: World's Greatest Heroes,2006–2007,MoonScoop Marvel Entertainment,Taffy Entertainment,Cartoon Network
5,Wolverine and the X-Men,2009,Toonz Entertainment Marvel Entertainment Marve...,Lionsgate Television,Nicktoons
6,M.O.D.O.K.,2021,Marvel Television,Disney Platform Distribution,Hulu
7,Hit-Monkey,2021–2024,Marvel Television,Disney Platform Distribution,Hulu


----------------------------------------

--- SAMPLE 2 ---
TITLE: Hydroelectric power stations in Nevada
STRATEGIES: ['FREQUENCY_MODE', 'AGGREGATION', 'UNIQUE_DIVERSITY']
DATA SNIPPET (First 2 rows):


,Name,Location,Capacity ( MW ),Operator,Year opened
0,Davis Dam,Clark County,251.0,United States Bureau of Reclamation,1951
1,Fleish,Washoe County,2.5,Truckee Meadows Water Authority,1905
2,Hoover Dam,Clark County,2080.0,United States Bureau of Reclamation,1936-1938/ 1944/1961
3,Lahontan Dam,Churchill County,5.9,Truckee-Carson Irrigation District,1915/1989
4,Verdi,Washoe County,2.3,Truckee Meadows Water Authority,1911
5,Washoe (Mogul),Washoe County,1.9,Truckee Meadows Water Authority,1904


----------------------------------------

--- SAMPLE 3 ---
TITLE: Earthquakes with the most property damage
STRATEGIES: ['FREQUENCY_MODE', 'UNIQUE_DIVERSITY', 'EXTREME_NUMERICAL']
DATA SNIPPET (First 2 rows):


...

----------------------------------------

--- SAMPLE 9 ---
TITLE: Institutes of Science and Technology in South Korea
STRATEGIES: ['FREQUENCY_MODE', 'UNIQUE_DIVERSITY', 'CATEGORICAL_FILTER_LOOKUP']
DATA SNIPPET (First 2 rows):


,Institution,Control,City,Regional
0,Daegu Gyeongbuk Institute of Science and Techn...,National (Corporate),Dalseong County,Daegu
1,Gwangju Institute of Science and Technology,National (Corporate),Buk District,Gwangju
2,Korea Advanced Institute of Science and Techno...,National (Corporate),Yuseong District,Daejeon
3,Ulsan National Institute of Science and Techno...,National (Corporate),Ulju County,Ulsan
4,University of Science and Technology,etc.,Yuseong District,Daejeon


----------------------------------------

--- SAMPLE 10 ---
TITLE: Pre-season matches of Stoke City F.C. in 2008–09
STRATEGIES: ['FREQUENCY_MODE', 'CATEGORICAL_FILTER_LOOKUP', 'UNIQUE_DIVERSITY']
DATA SNIPPET (First 2 rows):


,Match,Date,Opponent,Venue,Attendance
0,1,18 July 2008,Newcastle Town,A,3011
1,2,22 July 2008,1. FC Nürnberg,A,
2,3,25 July 2008,Bahrain,N,
3,4,29 July 2008,Notts County,A,2425
4,5,2 August 2008,Shrewsbury Town,A,2450
5,6,5 August 2008,Southampton,A,5040
6,7,8 August 2008,Osasuna,H,5419


----------------------------------------

